# Portfolio context

Original STAT 303-2 course analysis by Ben Saperstein, Winter 2026. Analytical cells and saved outputs are preserved below. Read [validation notes](../docs/methods-and-limitations.md) before interpreting model scores: turning speed contains the target, and a corrected evaluation has not been run. The original export is not included. The portfolio copy uses a portable Python kernel selector.


**Rules for the code:**

- Include all the code you used for your report in this file. The code for any section in the report should go under the same section in this file.
- Any missing code will result in -20% from its corresponding section in the report.
- Any irrelevant code will result in -20% from its corresponding section in the report.
- Make sure that you run your code before rendering so that all the necessary visual/numeric outputs are visible.
- Any code that is not properly run or throws errors will be considered missing/irrelevant.

## 4) Data

In [135]:
# =========================
# cstimer session36 → tidy DataFrame
# =========================

import json, re, numpy as np, pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

# More tools here
from sklearn.linear_model import Ridge, Lasso, LinearRegression, LogisticRegression, RidgeCV, LassoCV, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, accuracy_score, recall_score, roc_curve, auc, roc_auc_score, r2_score
from sklearn.model_selection import cross_val_score, cross_val_predict, cross_validate, train_test_split

import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import warnings
warnings.filterwarnings("ignore")

# ---- 1) Load JSON ----
filename = "cstimer_data.txt"   # <- change if needed
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

if "session36" not in data:
    raise KeyError(f"'session36' not found. Available keys (truncated): {list(data.keys())[:10]} ...")

session_data = data["session36"]  # list of solves

# ---- 2) Build base rows ----
rows = []
for idx, entry in enumerate(session_data):
    time_ms = scramble = timestamp = None
    events = []

    if isinstance(entry, list):
        # Typical: [ [0, time_ms], scramble, comment, timestamp, events? ]
        if len(entry) >= 1 and isinstance(entry[0], list) and len(entry[0]) >= 2:
            time_ms = entry[0][1]
        if len(entry) >= 2:
            scramble = entry[1]
        if len(entry) >= 4:
            timestamp = entry[3]
        if len(entry) >= 5 and isinstance(entry[4], list):
            events = entry[4]
    elif isinstance(entry, dict):
        time_ms   = entry.get("time_ms") or entry.get("time")
        scramble  = entry.get("scramble")
        timestamp = entry.get("timestamp") or entry.get("date")
        events    = entry.get("events", []) or []

    rows.append({
        "solve_index": idx,
        "time_ms": time_ms,
        "scramble": scramble,
        "timestamp": timestamp,
        "events": events
    })

solve_data = pd.DataFrame(rows)

# ---- 3) Date/Time split + weekday & time-of-day bucket ----
solve_data["datetime"] = pd.to_datetime(solve_data["timestamp"], unit="s", errors="coerce")
solve_data["date"] = solve_data["datetime"].dt.date
solve_data["time"] = solve_data["datetime"].dt.time
solve_data["day_of_week"] = solve_data["datetime"].dt.day_name()

def _bucket(hour: int) -> str:
    if 6 <= hour < 12:  return "Morning"
    if 12 <= hour < 18: return "Afternoon"
    if 18 <= hour < 24: return "Evening"
    return "Late Night"

solve_data["time_of_day_bucket"] = solve_data["datetime"].dt.hour.apply(lambda h: _bucket(int(h)) if pd.notna(h) else "Unknown")

# ---- 4) Move parsing & metrics ----
rot_pat  = re.compile(r"^\s*(x|y|z)(2|')?\s*$", re.IGNORECASE)
turn_pat = re.compile(
    r"^\s*(?:"
    r"(?:[RLUDFB](?:w)?|[MES])"      # face/wide/slice
    r"|2-2[RLUDFB]w"                 # multi-layer e.g., 2-2Lw
    r")(?:2|')?\s*$",
    re.IGNORECASE
)
move_rec_pat = re.compile(r"^\s*([^@]+)@(\d+)\s*$")
turn_token_pat = re.compile(
    r"^\s*((?:[RLUDFB](?:w)?|[MES]|2-2[RLUDFB]w))(2|')?\s*$",
    re.IGNORECASE
)

def parse_move_list(ev_list):
    """
    Robustly parse events that may come as ["R@100 U@200 ..."] (one long string)
    or as multiple strings. Returns:
      moves: [(token, t_ms)] sorted by time
      turns_ts: [t_ms] for turns
      rots_ts:  [t_ms] for rotations
      turn_tokens_in_order: [token] for turn moves in chronological order
    """
    if not isinstance(ev_list, list) or len(ev_list) == 0:
        return [], [], [], []

    # Flatten: split any string item into whitespace-separated tokens
    raw_tokens = []
    for e in ev_list:
        if isinstance(e, str):
            raw_tokens.extend([p for p in e.strip().split() if p])

    parsed = []
    for tok in raw_tokens:
        m = move_rec_pat.match(tok)
        if not m:
            continue
        move_token = m.group(1).strip()
        t = int(m.group(2))
        parsed.append((move_token, t))

    parsed.sort(key=lambda x: x[1])

    turns_ts = [t for tok, t in parsed if turn_pat.match(tok)]
    rots_ts  = [t for tok, t in parsed if rot_pat.match(tok)]
    turn_tokens_in_order = [tok for tok, _ in parsed if turn_pat.match(tok)]

    return parsed, turns_ts, rots_ts, turn_tokens_in_order

def inverse_turn_token(tok: str):
    """
    Inverse for turns (not rotations):
      R <-> R' , R2 -> R2 , Rw <-> Rw' , 2-2Lw <-> 2-2Lw'
    """
    if not isinstance(tok, str):
        return None
    m = turn_token_pat.match(tok.strip())
    if not m:
        return None
    base, mod = m.group(1), (m.group(2) or "")
    if mod == "2":
        inv_mod = "2"
    elif mod == "'":
        inv_mod = ""
    else:
        inv_mod = "'"
    return f"{base}{inv_mod}"

def longest_gap_ms(times_ms):
    if len(times_ms) < 2:
        return 0
    diffs = np.diff(times_ms)
    return int(diffs.max()) if len(diffs) else 0

def fastest_k_window_ms(times_ms, k=10):
    if len(times_ms) < k:
        return None
    best = None
    for i in range(len(times_ms) - k + 1):
        w = times_ms[i + k - 1] - times_ms[i]
        if best is None or w < best:
            best = w
    return int(best)

def per_solve_metrics(row):
    time_ms = row["time_ms"]
    events  = row["events"]

    moves, turns_ts, rots_ts, turn_tokens = parse_move_list(events)

    last_evt_ms = moves[-1][1] if moves else None
    if isinstance(time_ms, (int, float, np.integer, np.floating)) and time_ms and time_ms > 0:
        dur_ms = int(time_ms)
    else:
        dur_ms = last_evt_ms

    # Counts
    number_of_turns     = int(len(turns_ts))
    number_of_rotations = int(len(rots_ts))
    total_move_count    = int(len(moves))

    # Speed & pauses
    turn_speed_tps = (number_of_turns / (dur_ms / 1000.0)) if (dur_ms and dur_ms > 0) else np.nan
    longest_pause = longest_gap_ms([t for _, t in moves])
    fastest10_ms  = fastest_k_window_ms(turns_ts, k=10)

    # Scramble length
    scramble = row["scramble"]
    scramble_len = int(len(str(scramble).split())) if pd.notna(scramble) else 0

    # Wasted moves (turns only): immediate inverse pairs
    wasted = 0
    for i in range(len(turn_tokens) - 1):
        inv = inverse_turn_token(turn_tokens[i])
        if inv is not None and turn_tokens[i + 1] == inv:
            wasted += 1

    return pd.Series({
        "solve_time_s": (dur_ms / 1000.0) if dur_ms is not None else np.nan,   # float
        "number_of_turns": number_of_turns,                                     # int
        "turn_speed_tps": float(turn_speed_tps) if turn_speed_tps == turn_speed_tps else np.nan,
        "number_of_rotations": number_of_rotations,                             # int
        "total_move_count": total_move_count,                                   # int
        "longest_pause_ms": int(longest_pause),                                 # int
        "fastest_burst_10_moves_ms": fastest10_ms,                              # nullable int
        "wasted_moves": int(wasted),                                            # int
        "scramble_length": scramble_len                                         # int
    })

solve_data = pd.concat([solve_data, solve_data.apply(per_solve_metrics, axis=1)], axis=1)

# ---- 5) Final column order (day_of_week after date; then time; then bucket) ----
final_cols = [
    "solve_index",
    "solve_time_s",
    "date",
    "day_of_week",
    "time",
    "time_of_day_bucket",
    "number_of_turns",
    "turn_speed_tps",
    "number_of_rotations",
    "total_move_count",
    "longest_pause_ms",
    "fastest_burst_10_moves_ms",
    "wasted_moves",
    "scramble",
    "scramble_length",
]

solve_data_final = solve_data[final_cols].copy()

# ---- 6) Convert integers where possible; keep nullable Int64 for fastest_burst_10_moves_ms ----
int_cols_strict = [
    "solve_index", "number_of_turns", "number_of_rotations",
    "total_move_count", "longest_pause_ms", "wasted_moves", "scramble_length"
]
for c in int_cols_strict:
    solve_data_final[c] = pd.to_numeric(solve_data_final[c], errors="coerce").fillna(0).astype("int64")

solve_data_final["fastest_burst_10_moves_ms"] = pd.to_numeric(
    solve_data_final["fastest_burst_10_moves_ms"], errors="coerce"
).astype("Int64")  # remains NaN when fewer than 10 turns

# ---- 7) Set solve_index as the actual index ----
# solve_data_final = solve_data_final.set_index("solve_index", drop=True)

# Eliminating the outliers for both "longest_pause_ms" and "number_of_rotations"
solve_data_final = solve_data_final.loc[~((solve_data_final["longest_pause_ms"] > 3000) & (solve_data_final.index > 500))]
solve_data_final = solve_data_final.loc[~((solve_data_final["number_of_rotations"] > 25) & (solve_data_final.index > 500))]

# ---- 8) Done ----
display(solve_data_final.head())
print("\nRows:", len(solve_data_final), "Columns:", len(solve_data_final.columns))
print("\nDtypes:\n", solve_data_final.dtypes)
print()
print(solve_data_final.corr(numeric_only = True)['solve_time_s'])

# test_size = 0.2 imply that 20% of the data is used for testing and remaining 80% will be used for training

train_df, test_df = train_test_split(solve_data_final, test_size=0.2, random_state=2)

,solve_index,solve_time_s,date,day_of_week,time,time_of_day_bucket,number_of_turns,turn_speed_tps,number_of_rotations,total_move_count,longest_pause_ms,fastest_burst_10_moves_ms,wasted_moves,scramble,scramble_length
0,0,357.677,2020-12-22,Tuesday,17:40:32,Afternoon,316,0.883479,68,384,5321,3216,9,D R2 B2 U2 F2 D L2 D L2 U' L2 F D' R' F2 U' B ...,21
1,1,93.596,2020-12-22,Tuesday,17:46:42,Afternoon,119,1.271422,8,127,6927,2366,1,U' L2 F U' D B' L' R2 U L2 D R2 U' D' R2 F2 R2...,20
2,2,218.870,2020-12-22,Tuesday,17:49:35,Afternoon,133,0.607667,26,159,5319,4671,4,R D' U' F2 R2 D L2 B2 R2 B2 U F' D' B' U' L U'...,20
3,3,132.904,2020-12-22,Tuesday,17:53:40,Afternoon,90,0.677181,4,94,5201,6178,1,L2 F' U2 F D2 R2 B' R2 B F' D' L R2 D' R U B' ...,19
4,4,158.084,2020-12-22,Tuesday,17:57:22,Afternoon,109,0.689507,10,119,5395,5903,4,B' D L' F D B2 R L' B' F2 R2 D' B2 U' F2 R2 F2...,21



Rows: 22442 Columns: 15

Dtypes:
 solve_index                    int64
solve_time_s                 float64
date                          object
day_of_week                   object
time                          object
time_of_day_bucket            object
number_of_turns                int64
turn_speed_tps               float64
number_of_rotations            int64
total_move_count               int64
longest_pause_ms               int64
fastest_burst_10_moves_ms      Int64
wasted_moves                   int64
scramble                      object
scramble_length                int64
dtype: object

solve_index                 -0.320773
solve_time_s                 1.000000
number_of_turns              0.526393
turn_speed_tps              -0.584766
number_of_rotations          0.289350
total_move_count             0.577617
longest_pause_ms             0.671542
fastest_burst_10_moves_ms    0.662985
wasted_moves                 0.347860
scramble_length              0.009406
Name: solve_tim

## 5) Prediction

In [136]:

model_df = pd.DataFrame(columns=["Model Number", "New Predictor Variable(s)", "Training R^2", "Test R^2", "Training MAE", "Test MAE"])

In [137]:
X_train1 = train_df[['solve_index']]
y_train1 = train_df['solve_time_s']

X_test1 = test_df[['solve_index']]
y_test1 = test_df['solve_time_s']

model1 = LinearRegression()
model1.fit(X_train1, y_train1)

train1_pred = model1.predict(X_train1)
test1_pred = model1.predict(X_test1)

train1_R2 = r2_score(y_train1, train1_pred)
test1_R2 = r2_score(y_test1, test1_pred)

train1_mae = mean_absolute_error(y_train1, train1_pred)
test1_mae = mean_absolute_error(y_test1, test1_pred)

model_df.loc[0] = [1, "solve_index", train1_R2, test1_R2, train1_mae, test1_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589


In [138]:
X_train2 = train_df[['solve_index', 'turn_speed_tps']]
y_train2 = train_df['solve_time_s']

X_test2 = test_df[['solve_index', 'turn_speed_tps']]
y_test2 = test_df['solve_time_s']

model2 = LinearRegression()
model2.fit(X_train2, y_train2)

train2_pred = model2.predict(X_train2)
test2_pred = model2.predict(X_test2)

train2_R2 = r2_score(y_train2, train2_pred)
test2_R2 = r2_score(y_test2, test2_pred)

train2_mae = mean_absolute_error(y_train2, train2_pred)
test2_mae = mean_absolute_error(y_test2, test2_pred)

model_df.loc[1] = [2, "turn_speed_tps", train2_R2, test2_R2, train2_mae, train2_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.822237


In [139]:
X_train3 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms']]
y_train3 = train_df['solve_time_s']

X_test3 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms']]
y_test3 = test_df['solve_time_s']

model3 = LinearRegression()
model3.fit(X_train3, y_train3)

train3_pred = model3.predict(X_train3)
test3_pred = model3.predict(X_test3)

train3_R2 = r2_score(y_train3, train3_pred)
test3_R2 = r2_score(y_test3, test3_pred)

train3_mae = mean_absolute_error(y_train3, train3_pred)
test3_mae = mean_absolute_error(y_test3, test3_pred)

model_df.loc[2] = [3, "longest_pause_ms", train3_R2, test3_R2, train3_mae, test3_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.822237
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408


In [140]:
X_train4 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations']]
y_train4 = train_df['solve_time_s']

X_test4 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations']]
y_test4 = test_df['solve_time_s']

model4 = LinearRegression()
model4.fit(X_train4, y_train4)

train4_pred = model4.predict(X_train4)
test4_pred = model4.predict(X_test4)

train4_R2 = r2_score(y_train4, train4_pred)
test4_R2 = r2_score(y_test4, test4_pred)

train4_mae = mean_absolute_error(y_train4, train4_pred)
test4_mae = mean_absolute_error(y_test4, test4_pred)

model_df.loc[3] = [4, "number_of_rotations", train4_R2, test4_R2, train4_mae, test4_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.822237
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989


In [141]:
X_train5 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms']]
y_train5 = train_df['solve_time_s']

X_test5 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms']]
y_test5 = test_df['solve_time_s']

model5 = LinearRegression()
model5.fit(X_train5, y_train5)

train5_pred = model5.predict(X_train5)
test5_pred = model5.predict(X_test5)

train5_R2 = r2_score(y_train5, train5_pred)
test5_R2 = r2_score(y_test5, test5_pred)

train5_mae = mean_absolute_error(y_train5, train5_pred)
test5_mae = mean_absolute_error(y_test5, test5_pred)

model_df.loc[4] = [5, "fastest_burst_10_moves_ms", train5_R2, test5_R2, train5_mae, test5_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.822237
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989
4,5,fastest_burst_10_moves_ms,0.585575,0.671342,1.952847,1.908345


In [142]:
X_train6 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves']]
y_train6 = train_df['solve_time_s']

X_test6 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves']]
y_test6 = test_df['solve_time_s']

model6 = LinearRegression()
model6.fit(X_train6, y_train6)

train6_pred = model6.predict(X_train6)
test6_pred = model6.predict(X_test6)

train6_R2 = r2_score(y_train6, train6_pred)
test6_R2 = r2_score(y_test6, test6_pred)

train6_mae = mean_absolute_error(y_train6, train6_pred)
test6_mae = mean_absolute_error(y_test6, test6_pred)

model_df.loc[5] = [6, "wasted_moves", train6_R2, test6_R2, train6_mae, test6_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.822237
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989
4,5,fastest_burst_10_moves_ms,0.585575,0.671342,1.952847,1.908345
5,6,wasted_moves,0.640290,0.703395,1.800461,1.751248


In [143]:
X_train7 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves', 'day_of_week']]
y_train7 = train_df['solve_time_s']

X_test7 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves', 'day_of_week']]
y_test7 = test_df['solve_time_s']

X_train7_ohe = pd.get_dummies(X_train7, drop_first = True)
X_test7_ohe = pd.get_dummies(X_test7, drop_first = True)

model7 = LinearRegression()
model7.fit(X_train7_ohe, y_train7)

train7_pred = model7.predict(X_train7_ohe)
test7_pred = model7.predict(X_test7_ohe)

train7_R2 = r2_score(y_train7, train7_pred)
test7_R2 = r2_score(y_test7, test7_pred)

train7_mae = mean_absolute_error(y_train7, train7_pred)
test7_mae = mean_absolute_error(y_test7, test7_pred)

model_df.loc[6] = [7, "day_of_week", train7_R2, test7_R2, train7_mae, test7_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.822237
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989
4,5,fastest_burst_10_moves_ms,0.585575,0.671342,1.952847,1.908345
5,6,wasted_moves,0.640290,0.703395,1.800461,1.751248
6,7,day_of_week,0.640835,0.702479,1.800899,1.758624


In [144]:
X_train8 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves', 'day_of_week', 'time_of_day_bucket']]
y_train8 = train_df['solve_time_s']

X_test8 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves', 'day_of_week', 'time_of_day_bucket']]
y_test8 = test_df['solve_time_s']

X_train8_ohe = pd.get_dummies(X_train8, drop_first = True)
X_test8_ohe = pd.get_dummies(X_test8, drop_first = True)

model8 = LinearRegression()
model8.fit(X_train8_ohe, y_train8)

train8_pred = model8.predict(X_train8_ohe)
test8_pred = model8.predict(X_test8_ohe)

train8_R2 = r2_score(y_train8, train8_pred)
test8_R2 = r2_score(y_test8, test8_pred)

train8_mae = mean_absolute_error(y_train8, train8_pred)
test8_mae = mean_absolute_error(y_test8, test8_pred)

model_df.loc[7] = [8, "time_of_day_bucket", train8_R2, test8_R2, train8_mae, test8_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.822237
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989
4,5,fastest_burst_10_moves_ms,0.585575,0.671342,1.952847,1.908345
5,6,wasted_moves,0.640290,0.703395,1.800461,1.751248
6,7,day_of_week,0.640835,0.702479,1.800899,1.758624
7,8,time_of_day_bucket,0.640908,0.702498,1.800833,1.759127


In [145]:
poly = PolynomialFeatures(degree = 2, include_bias = False)
X_train_poly = poly.fit_transform(X_train8_ohe)
X_test_poly = poly.fit_transform(X_test8_ohe)

scaler = StandardScaler()
scaler.fit(X_train_poly)
X_train_poly_scaled = scaler.transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

rcv = RidgeCV(cv = 5)
rcv.fit(X_train_poly_scaled, y_train8)

train_poly_pred = rcv.predict(X_train_poly_scaled)
test_poly_pred = rcv.predict(X_test_poly_scaled)

train9_R2 = r2_score(y_train8, train_poly_pred)
test9_R2 = r2_score(y_test8, test_poly_pred)

train9_mae = mean_absolute_error(y_train8, train_poly_pred)
test9_mae = mean_absolute_error(y_test8, test_poly_pred)

model_df.loc[8] = [9, "non-linear terms", train9_R2, test9_R2, train9_mae, test9_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.822237
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989
4,5,fastest_burst_10_moves_ms,0.585575,0.671342,1.952847,1.908345
5,6,wasted_moves,0.640290,0.703395,1.800461,1.751248
6,7,day_of_week,0.640835,0.702479,1.800899,1.758624
7,8,time_of_day_bucket,0.640908,0.702498,1.800833,1.759127
8,9,non-linear terms,0.891035,0.834125,1.246007,1.241614


In [146]:
feature_names = poly.get_feature_names_out(X_train8_ohe.columns)

my_coefs = pd.DataFrame({"Feature": feature_names, "Coefficient": rcv.coef_})

ranked_coefs = my_coefs.reindex(my_coefs["Coefficient"].abs().sort_values(ascending=False).index)
display(ranked_coefs.head(10))
display(ranked_coefs.tail(10))

print("Best alpha:", rcv.alpha_)

,Feature,Coefficient
3,number_of_rotations,-4.779940
46,longest_pause_ms fastest_burst_10_moves_ms,3.900237
4,fastest_burst_10_moves_ms,3.562361
10,day_of_week_Tuesday,-3.309328
120,day_of_week_Tuesday^2,-3.309328
45,longest_pause_ms number_of_rotations,3.033454
69,fastest_burst_10_moves_ms^2,-2.716798
0,solve_index,-2.708180
75,fastest_burst_10_moves_ms day_of_week_Tuesday,2.480236
129,time_of_day_bucket_Evening^2,2.466151


,Feature,Coefficient
109,day_of_week_Sunday day_of_week_Tuesday,0.0
108,day_of_week_Sunday day_of_week_Thursday,0.0
102,day_of_week_Saturday day_of_week_Tuesday,0.0
101,day_of_week_Saturday day_of_week_Thursday,0.0
100,day_of_week_Saturday day_of_week_Sunday,0.0
95,day_of_week_Monday day_of_week_Wednesday,0.0
94,day_of_week_Monday day_of_week_Tuesday,0.0
93,day_of_week_Monday day_of_week_Thursday,0.0
92,day_of_week_Monday day_of_week_Sunday,0.0
91,day_of_week_Monday day_of_week_Saturday,0.0


Best alpha: 10.0


In [147]:
ranked_coefs.head(3)

,Feature,Coefficient
3,number_of_rotations,-4.779940
46,longest_pause_ms fastest_burst_10_moves_ms,3.900237
4,fastest_burst_10_moves_ms,3.562361


## 6) Inference

In [148]:
inf_model = smf.ols("solve_time_s ~ solve_index + time_of_day_bucket + longest_pause_ms*fastest_burst_10_moves_ms + longest_pause_ms*number_of_rotations + fastest_burst_10_moves_ms*day_of_week + I(fastest_burst_10_moves_ms ** 2)", data = train_df).fit()

inf_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           solve_time_s   R-squared:                       0.777
Model:                            OLS   Adj. R-squared:                  0.777
Method:                 Least Squares   F-statistic:                     2837.
Date:                Tue, 17 Mar 2026   Prob (F-statistic):               0.00
Time:                        00:00:07   Log-Likelihood:                -43496.
No. Observations:               17953   AIC:                         8.704e+04
Df Residuals:                   17930   BIC:                         8.722e+04
Df Model:                          22                                         
Covariance Type:            nonrobust                                         
======================================================================================================================
                                                         coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
Intercept                                             13.2732      0.420     31.604      0.000      12.450      14.096
time_of_day_bucket[T.Evening]                         -0.0317      0.056     -0.562      0.574      -0.142       0.079
time_of_day_bucket[T.Late Night]                      -0.0820      0.060     -1.368      0.171      -0.200       0.035
time_of_day_bucket[T.Morning]                         -0.0110      0.071     -0.155      0.877      -0.151       0.129
day_of_week[T.Monday]                                 -0.4204      0.461     -0.912      0.362      -1.324       0.483
day_of_week[T.Saturday]                                0.0094      0.569      0.016      0.987      -1.106       1.125
day_of_week[T.Sunday]                                 -1.6917      0.547     -3.091      0.002      -2.764      -0.619
day_of_week[T.Thursday]                               -0.0188      0.440     -0.043      0.966      -0.882       0.844
day_of_week[T.Tuesday]                                -6.4872      0.458    -14.170      0.000      -7.385      -5.590
day_of_week[T.Wednesday]                              -1.9541      0.434     -4.503      0.000      -2.805      -1.103
solve_index                                        -9.296e-05   4.11e-06    -22.632      0.000      -0.000   -8.49e-05
longest_pause_ms                                      -0.0061      0.000    -34.143      0.000      -0.006      -0.006
fastest_burst_10_moves_ms                              0.0018      0.000      3.759      0.000       0.001       0.003
fastest_burst_10_moves_ms:day_of_week[T.Monday]        0.0006      0.001      1.083      0.279      -0.000       0.002
fastest_burst_10_moves_ms:day_of_week[T.Saturday]      0.0001      0.001      0.173      0.862      -0.001       0.001
fastest_burst_10_moves_ms:day_of_week[T.Sunday]        0.0019      0.001      3.228      0.001       0.001       0.003
fastest_burst_10_moves_ms:day_of_week[T.Thursday]   6.342e-05      0.000      0.130      0.896      -0.001       0.001
fastest_burst_10_moves_ms:day_of_week[T.Tuesday]       0.0075      0.001     14.805      0.000       0.007       0.009
fastest_burst_10_moves_ms:day_of_week[T.Wednesday]     0.0024      0.000      4.999      0.000       0.001       0.003
longest_pause_ms:fastest_burst_10_moves_ms          4.262e-06   1.54e-07     27.598      0.000    3.96e-06    4.56e-06
number_of_rotations                                   -0.2900      0.008    -34.457      0.000      -0.306      -0.273
longest_pause_ms:number_of_rotations                   0.0005   6.54e-06     79.414      0.000       0.001       0.001
I(fastest_burst_10_moves_ms ** 2)                  -1.029e-06    1.6e-07     -6.418      0.000   -1.34e-06  